# KDD Process Volcano Data Analysis

In [45]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

In [46]:
def load_data(filepath="volcano-events.tsv"):
    try:
        df = pd.read_csv(filepath, sep='\t')
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return pd.DataFrame()

    df.rename(columns={
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    }, inplace=True)

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df.dropna(subset=['Year'], inplace=True)
    df['VEI'] = pd.to_numeric(df['VEI'], errors='coerce')
    df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
    df['Damage_Millions'] = pd.to_numeric(df['Damage_Millions'], errors='coerce').fillna(0)
    df.dropna(subset=['Latitude', 'Longitude', 'Country'], inplace=True)
    return df

## Spatial Analysis

In [ ]:
# Function: Build a scatter-based world map showing individual volcano eruptions.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least
#       ["Latitude", "Longitude", "Type", "Name", "VEI", "Country", "Year", "Deaths"]
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive scatter_geo map (used in Dash)

def get_map_points(df):
    # Copy the dataframe to avoid modifying the original one
    df_map = df.copy()

    # Replace missing VEI values with a small default value (0.5)
    # to avoid invisible points on the map
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    # Create the base geographic scatter plot
    fig = px.scatter_geo(
        df_map,
        lat="Latitude",           # latitude of volcano
        lon="Longitude",          # longitude of volcano
        color="Type",             # volcano morphological category
        size="VEI_Size",          # bubble size based on VEI (explosivity)
        hover_name="Name",        # volcano name in tooltip
        hover_data={              # additional tooltip information
            "Country": True,
            "Year": True,
            "Deaths": True,
            "VEI": True,
            "VEI_Size": False
        },
        title="Global Volcano Distribution (Bubble size = VEI)",
        projection="natural earth", # projection style
        size_max=15,                # maximum bubble size
        template="plotly_dark"      # dark theme to match dashboard
    )

    # Custom color palette: 20 vivid volcanic colors (orange → red → magenta → violet)
    warm_palette = [
   
    "#ffd500",  # deep orange
    "#ff8f00",  # vivid orange
    "#ff3d00",  # bright red-orange
    "#ff1a00",  # pure red-orange
    "#e60000",  # intense red
    "#c51162",  # magenta
    "#ff0055",  # neon pink-red
    "#d81b60",  # pink-magenta
    "#b0003a",  # dark magenta-red
    "#9c004d",  # deep pink-purple
    "#aa00ff",  # neon violet
    "#8e24aa",  # classic violet
    "#7b1fa2",  # deep violet
    "#6a1b9a",  # darker violet
    "#4a148c",  # almost purple-black
    "#7f0000",
    "#b30000",
    "#d50000",
    "#ff1744",
    "#ff4081"
    ]

    n_traces = len(fig.data)   # one trace per volcano Type
    n_colors = len(warm_palette)

    # Assign a distinct color from the palette to each volcano Type
    for i, trace in enumerate(fig.data):
        color = warm_palette[i % n_colors]     # loop through palette if Types > 20
        trace.marker.update(
            color=color,
            line=dict(width=0)                 # remove outline for cleaner look
        )

    # Add geographic features (coastlines, countries, land)
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True,
    )

    # Transparent background to match the dashboard's dark theme
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig

In [48]:
# Function: Build a choropleth (colored world map) aggregated by country.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least:
#       ["Country", value_col]
#   - value_col (str): column name used for coloring (e.g., "Count", "Deaths", "Damage_Millions")
#   - title (str): title of the choropleth
#   - color_label (str): label for the colorbar (e.g., "Eruptions", "Deaths", "Damage")
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive choropleth map (used in Dash)

def get_country_choropleth(df, value_col, title, color_label):
    # Aggregate data by country for the selected metric (count, deaths, damage…)
    agg = df.groupby('Country', as_index=False)[value_col].sum()
    # Custom continuous volcanic colormap (dark red → orange → magenta → violet)
    # Designed to avoid white/yellow and emphasize bright volcanic tones
    volcano_scale = [
        (0.00, "#000000"),   # very dark base
        (0.05, "#4b0000"),   # deep red
        (0.10, "#7f0000"),   # darker red
        (0.20, "#b00000"),   # intense red
        (0.30, "#d50000"),   # bright red
        (0.40, "#ff1400"),   # red-orange (flashy)
        (0.55, "#ff3d00"),   # bright orange-red
        (0.70, "#ff6d00"),   # orange incandescent
        (0.85, "#ff8500"),   # bright orange
        (0.93, "#d81b60"),   # magenta
        (1.00, "#6a1b9a")    # deep violet
    ]
    # Build the choropleth map
    fig = px.choropleth(
        agg,
        locations='Country',            # country name column
        locationmode='country names',   # match names to world countries
        color=value_col,                # metric used for color intensity
        hover_name='Country',           # tooltip title
        title=title,
        labels={value_col: color_label}, # name of the color axis
        template="plotly_dark",          # dark theme
        color_continuous_scale=volcano_scale,
        projection="natural earth"       # projection style
    )
    # Display country borders, coastlines, and land
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True
    )
    # Transparent background to blend with the dashboard
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig


## Temporal Analysis

In [49]:
def get_frequency_figure(df):
    fig = px.histogram(
        df, 
        x="Year", 
        title="Eruption Frequency",
        nbins=100,
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Year", 
        yaxis_title="Count",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Impact Analysis

In [50]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [51]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

In [52]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
            
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'} # Dropdown text needs to be black to be visible on white bg of default dropdown
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
       html.Div([
    # === BIG MAP CARD ===
    html.Div(
        [
            # Choix du type de carte
            html.Div([
                html.Label("Map view:", style={'color': 'white', 'margin-right': '10px'}),
                dcc.RadioItems(
                    id='map-mode',
                    options=[
                        {'label': 'Eruptions (points)', 'value': 'points'},
                        {'label': 'Eruptions / country', 'value': 'eruptions_country'},
                        {'label': 'Deaths / country', 'value': 'deaths_country'},
                        {'label': 'Damage / country', 'value': 'damage_country'},
                    ],
                    value='points',
                    inline=True,
                    className='dark-radio'
                )
            ], style={'margin-bottom': '10px'}),

            # La figure 
            dcc.Graph(id='map-graph', style={'height': '650px', 'width': '100%'})
        ],
        style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px', 'padding': '10px'}
    ),

    # === CARD DE DROITE (time graph comme avant) ===
    html.Div(
        [dcc.Graph(id='time-graph', style={'height': '650px', 'width': '100%'})],
        style={**CARD_STYLE, 'flex': '1', 'padding': '10px'}
    )
], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'})

    ], style=CONTENT_STYLE)
])

# Callbacks
@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('corr-graph', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value'),
     Input('map-mode', 'value')] 
     
)
def update_dashboard(selected_country, year_range, map_mode):
    dff = df.copy()

    # Filtre pays
    if selected_country:
        dff = dff[dff['Country'] == selected_country]

    # Filtre années
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # === CHOIX DE LA CARTE SELON map_mode ===
    if map_mode == 'points':
        fig_map = get_map_points(dff)

    elif map_mode == 'eruptions_country':
        dff_counts = dff.copy()
        dff_counts['Count'] = 1
        fig_map = get_country_choropleth(
            dff_counts, 'Count',
            title="Number of eruptions per country",
            color_label="Eruptions"
        )

    elif map_mode == 'deaths_country':
        fig_map = get_country_choropleth(
            dff, 'Deaths',
            title="Total deaths per country",
            color_label="Deaths"
        )

    elif map_mode == 'damage_country':
        fig_map = get_country_choropleth(
            dff, 'Damage_Millions',
            title="Total damage per country (Million USD)",
            color_label="Damage (M$)"
        )

    else:
        fig_map = get_map_points(dff)

    # Les autres figures comme avant
    fig_time = get_frequency_figure(dff)
    fig_impact = get_impact_figure(dff)
    fig_corr = get_correlation_figure(dff)

    return fig_map, fig_time, fig_impact, fig_corr, total_eruptions, total_deaths, total_damage

if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)

Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860
